# Transformers

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snss
import pandas as pd
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [2]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

GPU is available, using GPU Tesla T4


## Implementing scaled dot product attention

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math



seq = torch.rand(1, 3, 4)
print('Input sequence')
print(seq)
print()

W_q = torch.rand(2, 4)
W_k = torch.rand(2, 4)
W_v = torch.rand(2, 4)

# torch version
print('Output is')
F.scaled_dot_product_attention(
    seq @ W_q.T, seq @ W_k.T, seq @ W_v.T
)

Input sequence
tensor([[[0.3262, 0.3474, 0.3479, 0.4093],
         [0.2152, 0.1786, 0.8687, 0.4540],
         [0.7385, 0.8678, 0.4271, 0.4088]]])

Output is


tensor([[[0.8945, 1.0637],
         [0.9230, 1.0788],
         [0.9503, 1.0925]]])

In [4]:
def scaled_dot_product_attention(query,
                                 key,
                                 value,
                                 W_q,
                                 W_k,
                                 W_v):
    """ Query, Key, Value are batched sequences of vectors of size [B, N, F]
        W_q, W_k, W_v are matrices that map QKV to an embedding space of dim E
    """
    # Projection weights
    q = query @ W_q.T #[B, N, E]
    k = key @ W_k.T #[B, N, E]
    v = value @ W_v.T #[B, N, E]

    # Compute unnormalized attention score
    a_unnorm = torch.einsum('bie,bje->bij', q, k) / math.sqrt(q.shape[-1])

    # Normalize using softmax
    a_norm = F.softmax(a_unnorm, dim=-1)

    print('Aggregating with attention weights')
    print(a_norm)
    print()

    # Perform weighted sum over values
    output = torch.einsum('bij,bje->bie', a_norm, v)
    return output

result = scaled_dot_product_attention(seq, seq, seq, W_q, W_k, W_v)
print('Output is')
print(result)

Aggregating with attention weights
tensor([[[0.2179, 0.2299, 0.5522],
         [0.1842, 0.1988, 0.6171],
         [0.1540, 0.1661, 0.6799]]])

Output is
tensor([[[0.8945, 1.0637],
         [0.9230, 1.0788],
         [0.9503, 1.0925]]])


## Implementing multi-head self-attention

In [5]:
mha = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)

for param in mha.named_parameters():
    print(param)
    print(param[1].shape)

('in_proj_weight', Parameter containing:
tensor([[ 0.5955,  0.0801,  0.2969,  0.5444],
        [ 0.1695,  0.3289,  0.0894,  0.3726],
        [-0.0270,  0.4632, -0.2786, -0.1392],
        [-0.2266, -0.0248,  0.1173,  0.3655],
        [ 0.0910,  0.3565,  0.1600, -0.1484],
        [ 0.2467, -0.3823, -0.1149,  0.4857],
        [ 0.4190, -0.2035, -0.1245,  0.2179],
        [ 0.1342, -0.3215, -0.0223, -0.0859],
        [ 0.4725, -0.3221,  0.5517,  0.4478],
        [ 0.0768,  0.2516, -0.1159,  0.3749],
        [-0.0846, -0.2358, -0.4193,  0.3284],
        [-0.0713, -0.3574, -0.5472,  0.4018]], requires_grad=True))
torch.Size([12, 4])
('in_proj_bias', Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True))
torch.Size([12])
('out_proj.weight', Parameter containing:
tensor([[-0.3417,  0.0319, -0.1165, -0.3119],
        [-0.4406, -0.1486,  0.4813, -0.4188],
        [-0.2665, -0.3154,  0.4784,  0.1265],
        [-0.0978, -0.0140, -0.3870, -0.3838]], requ

In [6]:
outputs, attn_weights = mha(seq, seq, seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

Attention weights
tensor([[[0.3300, 0.3306, 0.3394],
         [0.3292, 0.3321, 0.3387],
         [0.3277, 0.3267, 0.3455]]], grad_fn=<MeanBackward1>)
Outputs
tensor([[[-0.0486, -0.2533, -0.3770,  0.1597],
         [-0.0487, -0.2533, -0.3769,  0.1595],
         [-0.0482, -0.2532, -0.3774,  0.1600]]], grad_fn=<TransposeBackward0>)


In [7]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, input_dim=4, embed_dim=4, num_heads=2):
        super().__init__()

        self.W_q = nn.Parameter(torch.rand(num_heads, embed_dim // num_heads, input_dim))
        self.W_k = nn.Parameter(torch.rand(num_heads, embed_dim // num_heads, input_dim))
        self.W_v = nn.Parameter(torch.rand(num_heads, embed_dim // num_heads, input_dim))

        self.W_o = nn.Parameter(torch.rand(embed_dim, embed_dim))

    def forward(self, seq):
        # Seq has shape [B, N, I]
        q = torch.einsum('bni,hei->bnhe', seq, self.W_q) # [B, N, H, E]
        k = torch.einsum('bni,hei->bnhe', seq, self.W_k) # [B, N, H, E]
        v = torch.einsum('bni,hei->bnhe', seq, self.W_v) # [B, N, H, E]

        a_unnorm = torch.einsum('bihe,bjhe->bhij', q, k) / math.sqrt(q.shape[-1])

        a_norm = F.softmax(a_unnorm, dim=-1)

        output_heads = torch.einsum('bhij,bjhe->bihe', a_norm, v)
        
        # Concatenate heads
        B, N, H, E = output_heads.shape
        output = output_heads.reshape([B, N, H*E])

        # Apply output projection matrix
        output = output @ self.W_o.T
        return output, a_norm.mean(dim=1)

mhsa = MultiHeadSelfAttention()
outputs, attn_weights = mhsa(seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

Attention weights
tensor([[[0.2636, 0.3205, 0.4160],
         [0.2428, 0.3174, 0.4398],
         [0.2129, 0.3020, 0.4851]]], grad_fn=<MeanBackward1>)
Outputs
tensor([[[2.0619, 1.1378, 2.8459, 2.5352],
         [2.0837, 1.1513, 2.8776, 2.5626],
         [2.1195, 1.1817, 2.9398, 2.6143]]], grad_fn=<UnsafeViewBackward0>)


In [8]:
W_q = mha.in_proj_weight[0:4, :].reshape([2, 2, 4])
W_k = mha.in_proj_weight[4:8, :].reshape([2, 2, 4])
W_v = mha.in_proj_weight[8:12, :].reshape([2, 2, 4])
W_o = mha.out_proj.weight

with torch.no_grad():
    mhsa.W_q[:] = W_q
    mhsa.W_k[:] = W_k
    mhsa.W_v[:] = W_v
    mhsa.W_o[:] = W_o

outputs, attn_weights = mhsa(seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

Attention weights
tensor([[[0.3300, 0.3306, 0.3394],
         [0.3292, 0.3321, 0.3387],
         [0.3277, 0.3267, 0.3455]]], grad_fn=<MeanBackward1>)
Outputs
tensor([[[-0.0486, -0.2533, -0.3770,  0.1597],
         [-0.0487, -0.2533, -0.3769,  0.1595],
         [-0.0482, -0.2532, -0.3774,  0.1600]]], grad_fn=<UnsafeViewBackward0>)


## PyTorch Transformer

In [9]:
import torch
import torch.nn as nn

transformer = nn.Transformer(d_model=16, nhead=4, num_encoder_layers=2, batch_first=True)
transformer

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=16, bias=True)
        (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_featu

## Dataset: Wikitext 2

In [10]:
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/train.txt
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/valid.txt
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/test.txt

File ‘/home/jcolen/scr10/wikitext2/train.txt’ already there; not retrieving.

File ‘/home/jcolen/scr10/wikitext2/valid.txt’ already there; not retrieving.

File ‘/home/jcolen/scr10/wikitext2/test.txt’ already there; not retrieving.



In [11]:
import os
from collections import Counter

def tokenize(line):
    words = line.split() + ['<eos>']
    return words

def build_vocab(text_path):
    vocab = Counter()
    with open(text_path, 'r') as f:
        for line in f:
            tokens = tokenize(line)
            vocab.update(tokens)

    vocab_enc = {
        '<pad>': 0,
        '<unk>': 1,
    }
    vocab_dec = ['<pad>', '<unk>']
    
    for token, count in sorted(vocab.items(), key=lambda x : x[1], reverse=True):
        if token == '<unk>': 
            continue
        vocab_enc[token] = len(vocab_enc)
        vocab_dec.append(token)
    

    return vocab_enc, vocab_dec

def encode_tokens(tokens, vocab_enc):
    return [ vocab_enc.get(token, vocab_enc['<unk>']) for token in tokens]

vocab_enc, vocab_dec = build_vocab(os.path.expanduser('~/scr10/wikitext2/train.txt'))
print(len(vocab_enc), len(vocab_dec))

33279 33279


In [12]:
# Really simple dataset
def encode_dataset(text_path, vocab_enc):
    with open(text_path, 'r') as f:
        all_tokens = []
        for line in f:
            tokens = tokenize(line)
            tokens = encode_tokens(tokens, vocab_enc)
            all_tokens.append(torch.LongTensor(tokens))
        all_tokens = torch.concat(all_tokens)

    return all_tokens

def shape_batches(data, batch_size=16):
    num_batches = data.shape[0] // batch_size
    data = data[:num_batches * batch_size]
    data = data.reshape([num_batches, batch_size])
    return data

batch_size = 16
train_data = encode_dataset(os.path.expanduser('~/scr10/wikitext2/train.txt'), vocab_enc)
train_data = shape_batches(train_data, batch_size)
print(train_data.shape)

from torch.utils.data import TensorDataset
train_dataset = TensorDataset(train_data)
val_dataset = TensorDataset(shape_batches(
    encode_dataset(
        os.path.expanduser('~/scr10/wikitext2/valid.txt'), vocab_enc
    ), batch_size
))

print(len(train_dataset), len(val_dataset))

torch.Size([130539, 16])
130539 13602


In [49]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, num_workers=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, num_workers=2, shuffle=False)

sample,  = next(iter(train_loader))
sample.shape

x = sample[:, :-1]
y = sample[:, 1:]

In [73]:
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=10000):
        super().__init__()
        pos_enc = torch.zeros([max_len, dim])
        pos = torch.arange(0, max_len, dtype=torch.float)[:, None]
        log_div = math.log(10000.) * torch.arange(0, dim, 2, dtype=torch.float)  / dim
        div = -log_div.exp()
        pos_enc[:, 0::2] = torch.sin(pos * div)
        pos_enc[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pos_enc', pos_enc) # Not a trainable parameter but still sent to GPU

    def forward(self, x):
        """ X has shape [B, L, E]
        """
        x = x + self.pos_enc[:x.shape[1], :]
        return x

def causal_mask(size, device):
    return torch.log(torch.tril(torch.ones(size, size, device=device)))

class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_size, nhead=2, num_layers=2):
        super().__init__()
        self.embed_size = embed_size
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.positional_encoding = PositionalEncoding(embed_size)
        self.transformer_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=embed_size, nhead=nhead, batch_first=True),
            num_layers=num_layers,
            norm=nn.LayerNorm(embed_size)
        )
        self.vocab_decoder = nn.Linear(embed_size, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x) * math.sqrt(self.embed_size)
        x = self.positional_encoding(x)
        mask = causal_mask(x.shape[1], x.device)
        memory = torch.zeros_like(x)
        output = self.transformer_decoder(tgt=x, memory=memory, tgt_mask=mask)
        output = self.vocab_decoder(output)
        return F.log_softmax(output, dim=-1)

model = Transformer(len(vocab_enc), 256, nhead=4, num_layers=2)
model = model.to(device)

y_pred = model(x.to(device))

print('Done!')
print(y_pred.shape)

Done!
torch.Size([64, 15, 33279])


In [51]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm, trange

def train(model, optimizer, loss_func, train_loader, val_loader, num_epochs=10):
    model = model.to(device)

    train_loss = []
    val_loss = []

    pbar_epoch = trange(num_epochs)
    for ii in pbar_epoch:
        model.train()
        train_loss_ = 0.
        train_correct_ = 0.
        for sample, in tqdm(train_loader, leave=False):
            # Send data to device
            xb = sample[:, :-1].to(device)
            yb = sample[:, 1:].to(device)

            # Make a prediction, compute the loss
            y_pred = model(xb)

            y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
            yb = yb.flatten()
            loss = loss_func(y_pred, yb)

            # Backpropagate, apply gradient updates, zero gradients
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            train_loss_ += loss.item() * yb.shape[0]
            
        train_loss.append(train_loss_ / len(train_loader.dataset))

        model.eval()
        with torch.no_grad():
            val_loss_ = 0.
            val_correct_ = 0.
            for xb, in tqdm(val_loader, leave=False):
                # Send data to device
                xb = sample[:, :-1].to(device)
                yb = sample[:, 1:].to(device)
    
                # Make a prediction, compute the loss
                y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
                yb = yb.flatten()
                loss = loss_func(y_pred, yb)

                # Logging
                val_loss_ += loss.item() * yb.shape[0]
            val_loss.append(val_loss_ / len(val_loader.dataset))

        pbar_epoch.set_postfix(
            epoch=f"{ii+1:03d} / {num_epochs:03d}",
            train_loss=f"{train_loss[-1]:.3f}",
            val_loss=f"{val_loss[-1]:.3f}")

    return model, (train_loss, val_loss)

# Define a loss function and optimizer
loss = nn.NLLLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model, model_stats = train(model, optimizer, loss, train_loader, val_loader, num_epochs=1)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/2040 [00:00<?, ?it/s]

  0%|          | 0/213 [00:00<?, ?it/s]

## Generate new words from a test sample

In [54]:
test_data = encode_dataset(os.path.expanduser('~/scr10/wikitext2/test.txt'), vocab_enc)
test_data

tensor([  9,  11, 613,  ...,   4,   9,   9])

In [67]:
def generate_sequence(seq_start, words=16):
    seq = []
    seq_input = seq_start.to(device)
    for ii in range(words):
        output = model(seq_input[None])[0]
        word_probs = output[-1].exp()
        word_idx = torch.multinomial(word_probs, 1)[0]
        seq_input = torch.concat([seq_input, torch.LongTensor([word_idx]).to(device)])
        seq.append(word_idx.item())

    return decode_tokens(seq)

def decode_tokens(tokens, vocab_dec=vocab_dec):
    return [ vocab_dec[ii] for ii in tokens ]

# Start from random token
seq_start = torch.randint(len(vocab_enc), size=[1,], dtype=torch.long).to(device)
print('Sequence start: ', ' '.join(decode_tokens(seq_start)))
print(' '.join(generate_sequence(seq_start)))

Sequence start:  Kamil
( @-@ ) ) ; mm . wanted the the quickly until The enough moved outer


In [69]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm, trange

def train(model, optimizer, loss_func, train_loader, val_loader, num_epochs=10):
    model = model.to(device)

    train_loss = []
    val_loss = []

    pbar_epoch = trange(num_epochs)
    for ii in pbar_epoch:
        model.train()
        train_loss_ = 0.
        train_correct_ = 0.
        for sample, in tqdm(train_loader, leave=False):
            # Send data to device
            xb = sample[:, :-1].to(device)
            yb = sample[:, 1:].to(device)

            # Make a prediction, compute the loss
            y_pred = model(xb)

            y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
            yb = yb.flatten()
            loss = loss_func(y_pred, yb)

            # Backpropagate, apply gradient updates, zero gradients
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            train_loss_ += loss.item() * yb.shape[0]
            
        train_loss.append(train_loss_ / len(train_loader.dataset))

        model.eval()
        with torch.no_grad():
            val_loss_ = 0.
            val_correct_ = 0.
            for xb, in tqdm(val_loader, leave=False):
                # Send data to device
                xb = sample[:, :-1].to(device)
                yb = sample[:, 1:].to(device)
    
                # Make a prediction, compute the loss
                y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
                yb = yb.flatten()
                loss = loss_func(y_pred, yb)

                # Logging
                val_loss_ += loss.item() * yb.shape[0]
            val_loss.append(val_loss_ / len(val_loader.dataset))

        pbar_epoch.set_postfix(
            epoch=f"{ii+1:03d} / {num_epochs:03d}",
            train_loss=f"{train_loss[-1]:.3f}",
            val_loss=f"{val_loss[-1]:.3f}")

        # Print out a sample sequence each epoch
        with torch.no_grad():
            seq_start = torch.randint(len(vocab_enc), size=[1,], dtype=torch.long).to(device)
            seq_extend = generate_sequence(seq_start)
            print(' '.join(decode_tokens(seq_start)), ' '.join(seq_extend))

    return model, (train_loss, val_loss)

# Define a loss function and optimizer
loss = nn.NLLLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model, model_stats = train(model, optimizer, loss, train_loader, val_loader, num_epochs=10)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2040 [00:00<?, ?it/s]

  0%|          | 0/213 [00:00<?, ?it/s]

drained they by are . to security <eos> <eos> <eos> 's are broadsides underlying His well in


  0%|          | 0/2040 [00:00<?, ?it/s]

...

  0%|          | 0/2040 [00:00<?, ?it/s]

  0%|          | 0/213 [00:00<?, ?it/s]

pyramid in Gold ; <unk> the of @-@ lower Interior 1995 <unk> of establishment the appointed Rederi


  0%|          | 0/2040 [00:00<?, ?it/s]

KeyboardInterrupt: 